# Milestone 4 — Transformers

Fine-tune DistilBERT for review rating-band classification and compare with M3 embeddings.

In [ ]:
import time
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score
from datasets import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)

from src.data import artifacts_dir, load_splits
from src.utils import load_json, save_json, setup_notebook_path

setup_notebook_path()

In [ ]:
MODEL_NAME = "distilbert-base-uncased"
MAX_SAMPLES = 12000  # keep training fast on free GPU

splits = load_splits()
train_r = splits["train_reviews"].head(MAX_SAMPLES).copy()
val_r = splits["val_reviews"].head(3000).copy()
test_r = splits["test_reviews"].head(3000).copy()

for df in (train_r, val_r, test_r):
    df["text"] = (df["review_title"] + " " + df["review_text"]).str.slice(0, 256)
    df["label"] = df["rating_band"] - 1

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=128)

train_ds = Dataset.from_pandas(train_r[["text", "label"]])
val_ds = Dataset.from_pandas(val_r[["text", "label"]])
test_ds = Dataset.from_pandas(test_r[["text", "label"]])

train_ds = train_ds.map(tokenize, batched=True)
val_ds = val_ds.map(tokenize, batched=True)
test_ds = test_ds.map(tokenize, batched=True)

cols = ["input_ids", "attention_mask", "label"]
train_ds.set_format(type="torch", columns=cols)
val_ds.set_format(type="torch", columns=cols)
test_ds.set_format(type="torch", columns=cols)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=5)

args = TrainingArguments(
    output_dir=str(artifacts_dir() / "distilbert_checkpoints"),
    evaluation_strategy="epoch",
    save_strategy="no",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=50,
    report_to="none",
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"macro_f1": f1_score(labels, preds, average="macro")}

trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds, compute_metrics=compute_metrics)

start = time.time()
trainer.train()
train_seconds = time.time() - start

pred = trainer.predict(test_ds)
preds = np.argmax(pred.predictions, axis=-1)
transformer_f1 = f1_score(test_ds["label"], preds, average="macro")
print("DistilBERT macro-F1:", transformer_f1)

In [ ]:
text_metrics = load_json(artifacts_dir() / "text_metrics.json") if (artifacts_dir() / "text_metrics.json").exists() else {}
emb_f1 = text_metrics.get("embedding_macro_f1")

save_json(
    {
        "distilbert_macro_f1": float(transformer_f1),
        "embedding_macro_f1": emb_f1,
        "train_seconds": train_seconds,
        "latency_note": "Transformers are slower but usually more accurate on short review text.",
    },
    artifacts_dir() / "transformer_metrics.json",
)
print("Comparison:", {"embedding": emb_f1, "distilbert": float(transformer_f1)})